<a href="https://colab.research.google.com/github/Dana-El/CCI-HW5/blob/main/Lab4_Memory_Persistence_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Memory & Persistence
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> A KHCC oncology nurse uses a clinical assistant throughout the day. The assistant needs to remember the conversation (short-term), resume if disconnected (checkpointing), and remember patient preferences across sessions (long-term memory).

### Objective
- Add short-term memory with checkpointing
- Use threads to scope conversations
- Implement long-term memory with stores
- Compare with Session 3's manual message list approach

---
## Setup

In [1]:
# Install required packages
!pip install -q langchain langchain-openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

# We'll use this model throughout the lab
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---
## Cell 1 -- Agent WITHOUT Memory (The Problem)

First, let's see what happens when an agent has **no memory**. Each call is completely independent.

In [4]:
# Create a simple agent with no memory
agent_no_memory = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful KHCC clinical assistant for oncology nurses."
)

# First question
response1 = agent_no_memory.invoke(
    {"messages": [{"role": "user", "content": "My patient Ahmad is 62 years old with Stage IIIA NSCLC."}]}
)
print("Response 1:")
print(response1["messages"][-1].content)
print("\n" + "="*60 + "\n")

# Second question -- the agent has NO context of the first question!
response2 = agent_no_memory.invoke(
    {"messages": [{"role": "user", "content": "What treatment options should I discuss with the oncologist for this patient?"}]}
)
print("Response 2 (no memory -- it forgot about Ahmad!):")
print(response2["messages"][-1].content)

/tmp/ipykernel_2032/4259200729.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_no_memory = create_react_agent(


Response 1:
Thank you for providing the information about your patient, Ahmad. Stage IIIA non-small cell lung cancer (NSCLC) typically indicates that the cancer has spread to nearby lymph nodes but has not metastasized to distant sites. Treatment options for Stage IIIA NSCLC often include a combination of surgery, chemotherapy, and radiation therapy, depending on the specific characteristics of the tumor and the patient's overall health.

Here are some considerations for managing Ahmad's care:

1. **Multidisciplinary Team Approach**: Ensure that Ahmad is evaluated by a multidisciplinary team, including medical oncologists, radiation oncologists, and thoracic surgeons, to determine the best treatment plan.

2. **Staging and Imaging**: Confirm the staging with appropriate imaging studies (CT scans, PET scans) to assess the extent of the disease and check for any distant metastasis.

3. **Treatment Options**:
   - **Surgery**: If the tumor is resectable, surgical options may include lobec

**Notice:** The agent doesn't know who "this patient" refers to. It lost all context!

In Session 3, we solved this by manually managing a message list. Now let's see LangGraph's approach.

---
## Cell 2 -- Add MemorySaver Checkpointer

LangGraph's `MemorySaver` automatically saves conversation state after each step. Using a `thread_id` scopes the conversation.

In [5]:
from langgraph.checkpoint.memory import MemorySaver

# TODO: Create a MemorySaver instance
checkpointer = MemorySaver()

# TODO: Create an agent WITH the checkpointer
agent_with_memory = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful KHCC clinical assistant for oncology nurses.",
    checkpointer=checkpointer
)

# TODO: Define a config with a thread_id to scope the conversation
config = {"configurable": {"thread_id": "thread-1"}}

# TODO: Ask the first question with the config
response1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "My patient Ahmad is 62 years old with Stage IIIA NSCLC."}]},
    config=config
)
print("Response 1:")
print(response1["messages"][-1].content)

# TODO: Ask a follow-up -- the agent should now remember!
response2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What treatment options should I discuss with the oncologist for this patient?"}]},
    config=config
)
print("\nResponse 2 (WITH memory -- it remembers Ahmad!):")
print(response2["messages"][-1].content)


/tmp/ipykernel_2032/2505847338.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_memory = create_react_agent(


Response 1:
Thank you for providing the information about your patient, Ahmad. Stage IIIA non-small cell lung cancer (NSCLC) typically indicates that the cancer has spread to nearby lymph nodes but has not metastasized to distant sites. Treatment for Stage IIIA NSCLC often involves a combination of therapies, which may include:

1. **Surgery**: If the tumor is resectable, surgical removal of the tumor and affected lymph nodes may be considered.

2. **Chemotherapy**: Neoadjuvant chemotherapy (before surgery) or adjuvant chemotherapy (after surgery) is often used to reduce the risk of recurrence.

3. **Radiation Therapy**: This may be used in conjunction with chemotherapy, especially if surgery is not an option or to target remaining cancer cells post-surgery.

4. **Targeted Therapy or Immunotherapy**: Depending on the specific characteristics of the tumor (e.g., presence of mutations like EGFR or ALK), targeted therapies or immunotherapy may be appropriate.

5. **Clinical Trials**: Part

---
## Cell 3 -- Multiple Threads (Different Patients)

Each `thread_id` creates an **isolated** conversation. Perfect for tracking different patients!

In [6]:
# TODO: Create two separate thread configs for two different patients
config_patient_1 = {"configurable": {"thread_id": "patient-18887"}}
config_patient_2 = {"configurable": {"thread_id": "patient-21307"}}

# TODO: Talk about Patient 1 (Ahmad, NSCLC)
response = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Patient Ahmad, MRN 18887, has Stage IIIA NSCLC. He is on carboplatin/paclitaxel."}]},
    config=config_patient_1
)
print("Patient 18887 thread:")
print(response["messages"][-1].content)

# TODO: Talk about Patient 2 (Sara, breast cancer)
response = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Patient Sara, MRN 21307, has Stage IIA breast cancer. She is on AC-T regimen."}]},
    config=config_patient_2
)
print("\nPatient 21307 thread:")
print(response["messages"][-1].content)

# TODO: Ask about "the patient's chemo regimen" in each thread
# Show that each thread correctly identifies the right patient
response_p1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is this patient's current chemo regimen?"}]},
    config=config_patient_1
)
response_p2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is this patient's current chemo regimen?"}]},
    config=config_patient_2
)
print("\n" + "="*60)
print("Thread patient-18887 answer:", response_p1["messages"][-1].content[:100])
print("Thread patient-21307 answer:", response_p2["messages"][-1].content[:100])


Patient 18887 thread:
Thank you for the information about patient Ahmad. As a clinical assistant, I can help you with various aspects of his care. Here are some considerations and recommendations for managing a patient with Stage IIIA Non-Small Cell Lung Cancer (NSCLC) on carboplatin and paclitaxel:

1. **Monitoring for Side Effects**: 
   - **Hematologic**: Monitor for signs of myelosuppression, including anemia, thrombocytopenia, and neutropenia. Regular complete blood counts (CBC) should be performed.
   - **Gastrointestinal**: Watch for nausea, vomiting, and diarrhea. Antiemetics may be needed.
   - **Neurological**: Assess for peripheral neuropathy, which can be a side effect of paclitaxel.
   - **Allergic Reactions**: Be vigilant for hypersensitivity reactions, especially with paclitaxel.

2. **Supportive Care**:
   - Ensure that Ahmad has access to antiemetics and other supportive medications as needed.
   - Encourage hydration and nutritional support, especially if he experienc

---
## Cell 4 -- Resume After "Disconnection" (Checkpointing)

The real power of checkpointing: if the nurse's session drops, they can **resume** exactly where they left off.

In [7]:
# TODO: Simulate a "disconnection" by creating a BRAND NEW agent instance
# but reusing the SAME checkpointer and thread_id

# This simulates: nurse closes browser, reopens later
# The old agent_with_memory variable is "gone" -- we make a new one

new_agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful KHCC clinical assistant for oncology nurses.",
    checkpointer=checkpointer   # <-- same checkpointer!
)

# TODO: Resume the conversation for patient-18887
# Use the same thread_id as before
config_resume = {"configurable": {"thread_id": "patient-18887"}}

response = new_agent.invoke(
    {"messages": [{"role": "user", "content": "Has this patient had any imaging done recently?"}]},
    config=config_resume
)
print("Resumed conversation -- agent still knows about Ahmad!")
print(response["messages"][-1].content)

# TODO: Inspect the full conversation history stored in the checkpoint
state = new_agent.get_state(config_resume)
print(f"\nTotal messages in checkpoint: {len(state.values['messages'])}")
for msg in state.values['messages']:
    print(f"  {msg.type}: {msg.content[:80]}...")


/tmp/ipykernel_2032/3424356375.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  new_agent = create_react_agent(


Resumed conversation -- agent still knows about Ahmad!
As a clinical assistant, I do not have access to real-time patient records or imaging results. To determine if patient Ahmad has had any recent imaging done, you would need to check his medical records or consult with the oncology team responsible for his care.

Typically, for a patient with Stage IIIA NSCLC undergoing chemotherapy, follow-up imaging (such as a CT scan or PET scan) is often performed after a few cycles of treatment to assess the response to therapy. If you have access to the electronic health record (EHR) system, you can look for any recent imaging reports or scheduled imaging appointments for Ahmad.

If you need assistance with how to access that information or have any other questions, feel free to ask!

Total messages in checkpoint: 6
  human: Patient Ahmad, MRN 18887, has Stage IIIA NSCLC. He is on carboplatin/paclitaxel....
  ai: Thank you for the information about patient Ahmad. As a clinical assistant, I ca.

---
## Cell 5 -- Long-Term Memory with InMemoryStore

Checkpointing is **short-term** (within a thread). But what about information that should persist **across all threads**? That's where `InMemoryStore` comes in.

Think: patient preferences, allergies, language -- things that matter in every conversation.

In [8]:
from langgraph.store.memory import InMemoryStore

# TODO: Create a store for long-term patient preferences
store = InMemoryStore()

# TODO: Pre-populate the store with patient preferences
# The store uses namespaces (like folders) and keys
store.put(
    namespace=("patients", "18887", "preferences"),
    key="language",
    value={"preference": "Arabic", "note": "Patient prefers all communications in Arabic"}
)
store.put(
    namespace=("patients", "18887", "preferences"),
    key="allergies",
    value={"allergy": "Penicillin", "severity": "severe", "reaction": "anaphylaxis"}
)
store.put(
    namespace=("patients", "18887", "preferences"),
    key="contact",
    value={"preferred_contact": "son", "son_name": "Khaled", "son_phone": "+962-79-XXX-XXXX"}
)

# TODO: Create a tool that reads from the store
@tool
def get_patient_preferences(patient_mrn: str) -> str:
    """Look up stored preferences for a patient by MRN number."""
    items = store.search(("patients", patient_mrn, "preferences"))
    if not items:
        return f"No preferences stored for patient {patient_mrn}"
    prefs = []
    for item in items:
        prefs.append(f"{item.key}: {item.value}")
    return "\n".join(prefs)

# TODO: Create an agent with this tool AND the checkpointer
agent_with_store = create_react_agent(
    model=model,
    tools=[get_patient_preferences],
    prompt="You are a KHCC clinical assistant. Always check patient preferences before giving advice. Use the get_patient_preferences tool with the patient's MRN.",
    checkpointer=checkpointer
)

# TODO: Test -- the agent should look up preferences from the store
config_new_thread = {"configurable": {"thread_id": "nurse-morning-shift"}}
response = agent_with_store.invoke(
    {"messages": [{"role": "user", "content": "I need to prepare for patient MRN 18887's appointment. What should I know?"}]},
    config=config_new_thread
)
print(response["messages"][-1].content)


/tmp/ipykernel_2032/3154734993.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_store = create_react_agent(


For patient MRN 18887, here are the important details to prepare for their appointment:

1. **Language Preference**: The patient prefers all communications in Arabic.

2. **Allergies**: The patient has a severe allergy to Penicillin, which can cause anaphylaxis. Ensure that this is noted in their medical records and that all staff are aware.

3. **Contact Information**: The preferred contact person is the patient's son, Khaled. His phone number is +962-79-XXX-XXXX.

Make sure to communicate with the patient in Arabic and take necessary precautions regarding their allergy.


---
## Cell 6 -- Context Trimming

As conversations grow long, we risk exceeding the model's context window (and paying more!). LangGraph supports trimming strategies.

In [9]:
from langchain_core.messages import trim_messages

# TODO: Create a trimming function that keeps only the last N messages
trimmer = trim_messages(
    max_tokens=500,
    strategy="last",
    token_counter=model,
    include_system=True,       # Always keep the system prompt
    allow_partial=False,
)

# TODO: Simulate a long conversation and show trimming in action
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

long_conversation = [
    SystemMessage(content="You are a KHCC clinical assistant."),
    HumanMessage(content="Patient Ahmad has NSCLC."),
    AIMessage(content="I understand. Stage and current treatment?"),
    HumanMessage(content="Stage IIIA, on carboplatin/paclitaxel cycle 3."),
    AIMessage(content="Noted. How is he tolerating the regimen?"),
    HumanMessage(content="Moderate nausea, grade 2. We added ondansetron."),
    AIMessage(content="Good choice. Monitor for QT prolongation with ondansetron."),
    HumanMessage(content="His ANC dropped to 1.2 yesterday."),
    AIMessage(content="That's concerning. Consider G-CSF and recheck in 48 hours."),
    HumanMessage(content="Should we delay the next cycle?"),
]

trimmed = trimmer.invoke(long_conversation)
print(f"Original messages: {len(long_conversation)}")
print(f"Trimmed messages: {len(trimmed)}")
print("\nKept messages:")
for msg in trimmed:
    print(f"  {msg.type}: {msg.content[:60]}")

# Compare with Session 3: In Session 3 we manually counted tokens.
# LangGraph handles this automatically with trim_messages!


Original messages: 10
Trimmed messages: 10

Kept messages:
  system: You are a KHCC clinical assistant.
  human: Patient Ahmad has NSCLC.
  ai: I understand. Stage and current treatment?
  human: Stage IIIA, on carboplatin/paclitaxel cycle 3.
  ai: Noted. How is he tolerating the regimen?
  human: Moderate nausea, grade 2. We added ondansetron.
  ai: Good choice. Monitor for QT prolongation with ondansetron.
  human: His ANC dropped to 1.2 yesterday.
  ai: That's concerning. Consider G-CSF and recheck in 48 hours.
  human: Should we delay the next cycle?


---
## Cell 7 -- Comparison with Session 3

Let's compare Session 3's manual approach vs LangGraph's built-in memory.

In [10]:
# SESSION 3 APPROACH (manual message list)
print("=" * 60)
print("SESSION 3: Manual Message Management")
print("=" * 60)
print("""
messages = []
messages.append({"role": "user", "content": "..."})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages        # <-- manually pass full history
)
messages.append(response)   # <-- manually append response

Problems:
  - YOU manage the message list
  - YOU handle token counting / trimming
  - No persistence if Python session crashes
  - No thread isolation (you build it yourself)
  - No long-term memory across sessions
""")

print("=" * 60)
print("SESSION 5: LangGraph Memory & Persistence")
print("=" * 60)
print("""
checkpointer = MemorySaver()
agent = create_react_agent(model, tools, checkpointer=checkpointer)

# Just invoke with a thread_id -- everything is automatic!
agent.invoke(
    {"messages": [{"role": "user", "content": "..."}]},
    config={"configurable": {"thread_id": "patient-18887"}}
)

Benefits:
  + Automatic message management
  + Built-in trimming (trim_messages)
  + Survives crashes (checkpointing)
  + Thread isolation built-in
  + Long-term memory with InMemoryStore
  + Production: swap MemorySaver for PostgresSaver
""")

SESSION 3: Manual Message Management

messages = []
messages.append({"role": "user", "content": "..."})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages        # <-- manually pass full history
)
messages.append(response)   # <-- manually append response

Problems:
  - YOU manage the message list
  - YOU handle token counting / trimming
  - No persistence if Python session crashes
  - No thread isolation (you build it yourself)
  - No long-term memory across sessions

SESSION 5: LangGraph Memory & Persistence

checkpointer = MemorySaver()
agent = create_react_agent(model, tools, checkpointer=checkpointer)

# Just invoke with a thread_id -- everything is automatic!
agent.invoke(
    {"messages": [{"role": "user", "content": "..."}]},
    config={"configurable": {"thread_id": "patient-18887"}}
)

Benefits:
  + Automatic message management
  + Built-in trimming (trim_messages)
  + Survives crashes (checkpointing)
  + Thread isolation built-in
  + Lo

---
## Stretch Goals

If you finish early, try these:

1. **Add a `save_patient_preference` tool** that writes to the store, so the agent can learn new preferences during conversation.
2. **Implement a summary-based trimming strategy**: instead of just dropping old messages, summarize them first.
3. **Count the tokens** in a checkpoint's message history using `model.get_num_tokens()`.

---
## KHCC Connection

At KHCC, clinical assistants would need:
- **Short-term memory**: nurse asks follow-up questions within a shift
- **Checkpointing**: nurse's session drops during a busy shift, resumes seamlessly
- **Thread isolation**: separate conversations per patient (MRN-based threads)
- **Long-term memory**: patient allergies, language preferences, family contacts persist across all interactions
- **Production deployment**: replace `MemorySaver` with `PostgresSaver` backed by the hospital's database

This is the foundation for building a real clinical AI assistant at KHCC!